# gu_toolkit overview: concise notebook patterns

Run the setup cell once, then jump to the section you need and tweak the sliders.


In [1]:
!pip -q install plotly pandas scipy numpy sympy anywidget
import import_setup
from gu_toolkit import *

## 1) Quickstart

The shortest useful pattern is: create a figure, display it, then use module-level helpers inside `with fig:` so the code stays compact.

In [2]:
fig = Figure(x_range=(0, 1/220), y_range=(-2.5, 2.5), samples=1200)
fig.show()

with fig:
    set_title("Quickstart: one expression, two sliders")
    plot(a * sin(2*pi*b * x), x, label=r"$a\sin(2 \pi b x)$", color="royalblue")
    parameter(a, value=2.0, min=0.2, max=1.9, step=0.1)
    parameter(b, value=220.0, min=50, max=440.0, step=0.1)

OneShotOutput()

In [3]:
help(fig.parameters["a"])
help(fig.parameter)
help(parameter)
#BUG: the help for ProxyParamRef and  fig.parameter is very badly organized. 
# It is excessively long, contains irrelevant and generic discoverability links and does not efficiently illustrate main use cases and necessary info for the end user

Help on ProxyParamRef in module gu_toolkit.ParamRef object:

class ProxyParamRef(builtins.object)
 |  ProxyParamRef(parameter: 'Symbol', widget: 'Any') -> 'None'
 |
 |  Default ParamRef implementation that proxies to a widget/control.
 |
 |  Full API
 |  --------
 |  ``ProxyParamRef(parameter: Symbol, widget: Any)``
 |
 |  Public members exposed from this class: ``parameter``, ``widget``, ``value``, ``default_value``, ``observe``, ``reset``,
 |      ``capabilities``, ``min``, ``max``, ``step``, ``animation_time``,
 |      ``animation_mode``, ``animation_running``, ``start_animation``, ``stop_animation``,
 |      ``toggle_animation``
 |
 |  Parameters
 |  ----------
 |  parameter : Symbol
 |      Parameter symbol or parameter reference associated with this API. Required.
 |
 |  widget : Any
 |      Widget/control instance associated with this API. Required.
 |
 |  Returns
 |  -------
 |  ProxyParamRef
 |      New ``ProxyParamRef`` instance configured according to the constructor argumen

## 2) Numeric callables

Use the callable-first API when your model already exists as NumPy code. The only rule is: use NumPy operations inside the callable so array inputs evaluate correctly.

In [4]:
fig_callable = Figure(x_range=(-8, 8), y_range=(-2.5, 2.5), samples=800)
fig_callable.show()

with fig_callable:
    set_title("Callable-first API: reuse numeric code")
    plot(
        lambda x, A, k: A * np.exp(-0.1 * x**2) * np.cos(k * x),
        x,
        vars=(x, A, k),
        label=r"$A e^{-0.1x^2}\cos(kx)$",
        color="teal",
    )
    plot(
        lambda x, A: A / (1 + x**2),
        x,
        vars=(x, A),
        label=r"$A/(1+x^2)$",
        color="orange",
        opacity=0.45,
    )
    parameter(A, value=1.2, min=0.2, max=2.0, step=0.1)
    parameter(k, value=2.5, min=0.5, max=6.0, step=0.1)

OneShotOutput()

## 3) Info cards

Info cards are useful when you want the plot and the explanation to live in the same widget. A card can be plain HTML, or a callback that recomputes derived quantities whenever the figure re-renders.

In [5]:
fig_info = Figure(x_range=(-2 * pi, 2 * pi), y_range=(-3, 3), samples=900)
fig_info.show()

with fig_info:
    set_title("Info cards can mix static notes with live diagnostics")
    plot(a * sin(b * x), x, label=r"$a\sin(bx)$", color="royalblue")
    parameter(a, value=1.0, min=0.2, max=2.0, step=0.1)
    parameter(b, value=1.0, min=0.5, max=4.0, step=0.1)

    info(
        "<b>Reading guide</b><br>Move <code>a</code> to scale the wave and <code>b</code> to change frequency.",
        id="usage",
    )

    def diagnostics(fig, ctx):
        p = fig.parameters.snapshot()
        a, b = p["a"], p["b"]
        return (
            f"<b>Diagnostics</b><br>"
            f"render reason: <code>{ctx.reason}</code><br>"
            f"period ≈ {2 * pi / b:.3f}<br>"
            f"energy proxy $a^2/2$ ≈ {0.5 * a**2:.3f}"
        )

    info(diagnostics, id="diagnostics")

OneShotOutput()

## 4) Multiple views

Views are useful when several related pictures should share the same parameters. Here the main view shows a shifted sine, while the second view adds a reference cosine.

In [6]:
fig_views = Figure(x_range=(-2 * pi, 2 * pi), y_range=(-3, 3))
fig_views.add_view("phase", title="Phase view", x_range=(-2 * pi, 2 * pi), y_range=(-3, 3), x_label="phase")
fig_views.show()

with fig_views:
    set_title("One figure, multiple views")
    plot(a * sin(x) + b, x, label=r"$a\sin(x)+b$", view=("main", "phase"), color="crimson")
    parameter(a, value=1.0, min=0.0, max=2.0, step=0.1)
    parameter(b, value=0.5, min=-2.0, max=2.0, step=0.1)
    info("<b>Shared card</b><br>This card appears on every view.", id="shared-card")
    info("<b>Phase note</b><br>The second view can carry its own plots and cards.", id="phase-card", view="phase")

    with fig_views.views["phase"]:
        plot(cos(2 * x), x, label=r"$\cos(2x)$", color="purple")

OneShotOutput()

## 5) Parametric plots

Parametric plots are the compact way to describe curves that are easier to think about as `(x(t), y(t))` than as `y = f(x)`. The first example is a slider-controlled Lissajous family.

In [7]:
phase = sp.symbols("phase")

fig_param = Figure(x_range=(-1.2, 1.2), y_range=(-1.2, 1.2), samples=1500)
fig_param.show()

with fig_param:
    set_title("Parametric plot: a Lissajous family")
    parametric_plot(
        (sin(3 * 2 * pi * t + phase), sin(2 * 2 * pi * t)),
        (t, 0, 1),
        id="lissajous",
        label="lissajous",
        color="purple",
        samples=2400,
    )
    parametric_plot(
        (cos(2 * pi * t), sin(2 * pi * t)),
        (t, 0, 1),
        id="reference-circle",
        label="unit circle",
        color="gray",
        opacity=0.2,
        samples=300,
    )
    parameter(phase, value=0.0, min=0.0, max=float(2 * np.pi), step=0.05)
    info(
        "<b>What to watch</b><br>As <code>phase</code> changes, the curve slides through different resonances while keeping the same two frequencies.",
        id="param-note",
    )

OneShotOutput()

A second parametric example uses a modulated radius. This is the pattern behind many rose-curve and spirograph-style explorations: define a radius once, then multiply by `(cos, sin)`.

In [8]:
petals, twist = symbols(r"petals twist")
r = 1 + 0.35 * cos(petals * 2 * pi * t + twist * t)

fig_param_rose = Figure(x_range=(-1.5, 1.5), y_range=(-1.5, 1.5))
fig_param_rose.show()

with fig_param_rose:
    set_title("Parametric plot: polar-style modulation")
    parametric_plot(
        (r * cos(2 * pi * t), r * sin(2 * pi * t)),
        (t, 0, 1),
        id="rose",
        label="breathing rose",
        color="crimson",
        samples=2600,
    )
    parameter(petals, value=5, min=2, max=10, step=1)
    parameter(twist, value=0.0, min=0.0, max=18.0, step=0.2)
    info(
        "<b>Pattern</b><br>This is still one compact symbolic definition: a radius modulation multiplied by <code>(cos, sin)</code>.",
        id="rose-note",
    )

OneShotOutput()

## 6) Contour and temperature plots

For scalar fields, contour plots emphasize **level sets** and topology, while temperature plots emphasize **magnitude**. The next figure puts the same field into two views so you can compare them directly.

In [10]:
cx, cy, sigma = sp.symbols("cx cy sigma")
field_expr = (
    exp(-((x - cx) ** 2 + (y - cy) ** 2) / (2 * sigma**2))
    - 0.7 * exp(-((x + 0.8 * cx) ** 2 + (y + 0.4 * cy) ** 2) / (2 * (1.6 * sigma) ** 2))
)

fig_field = Figure(x_range=(-3, 3), y_range=(-3, 3))
fig_field.add_view("temperature", title="Temperature", x_range=(-3, 3), y_range=(-3, 3))
fig_field.show()

with fig_field:
    set_title("Same scalar field, two render modes")
    contour(field_expr, x, y, id="field-contours", label="contours", grid=(80, 80), levels=14, line_width=2)
    temperature(
        field_expr,
        x,
        y,
        id="field-temperature",
        label="temperature",
        view="temperature",
        grid=(80, 80),
        show_colorbar=True,
    )
    parameter(cx, value=0.8, min=-1.5, max=1.5, step=0.1)
    parameter(cy, value=-0.4, min=-1.5, max=1.5, step=0.1)
    parameter(sigma, value=0.8, min=0.4, max=1.6, step=0.1)
    info("<b>Main view</b><br>Contours emphasize level sets and topology.", id="field-main-note")
    info(
        "<b>Temperature view</b><br>Switch views to see the same field as a heatmap.",
        id="field-heat-note",
        view="temperature",
    )

OneShotOutput()

Contour lines also work well as an overlay. This is a good pattern when you want a single figure to show both intensity and structure.

In [11]:
stretch = symbols(r"stretch")
overlay_expr = sin(stretch * x) * cos(1.5 * y) + 0.15 * (x**2 - y**2)

fig_overlay = Figure(x_range=(-3, 3), y_range=(-3, 3))
fig_overlay.show()

with fig_overlay:
    set_title("Temperature map with contour overlay")
    temperature(overlay_expr, x, y, id="overlay-temp", grid=(90, 90), show_colorbar=True, opacity=0.9)
    contour(
        overlay_expr,
        x,
        y,
        id="overlay-contours",
        grid=(90, 90),
        levels=12,
        line_color="white",
        line_width=1.5,
    )
    parameter(stretch, value=1.2, min=0.5, max=2.5, step=0.1)

OneShotOutput()

## 7) Symbolic functions, integration, and Fourier coefficients

The pattern below defines reusable symbolic waveforms once, plots them, and then extracts many Fourier coefficients with `NReal_Fourier_Series(...)` instead of writing the integration loop by hand.

In [27]:
@NamedFunction
def Sq(x):
    return sign(sin(2 * pi * x))


@NamedFunction
def Tr(x):
    return asin(sin(2 * pi * x)) / (pi / 2)


fig_waveforms = Figure(x_range=(-1, 1), y_range=(-1.2, 1.2), samples=1200)
fig_waveforms.show()

with fig_waveforms:
    set_title("Two periodic shapes used below")
    plot(Sq(x), x, label=r"$\mathrm{Sq}(x)$", color="royalblue")
    plot(Tr(x), x, label=r"$\mathrm{Tr}(x)$", color="darkorange")

OneShotOutput()

In [28]:
def show_real_fourier_coefficients(title, expr, harmonics=12, samples=4096):
    # TODO(helper): notebook examples keep rebuilding this small coefficient table;
    # a built-in toolkit display helper would keep the notebook focused on the math.
    cos_coeffs, sin_coeffs = NReal_Fourier_Series(expr, (x, -1 / 2, 1 / 2), samples=samples)
    df = pd.DataFrame(
        {
            "n": np.arange(harmonics + 1),
            "cos_n": cos_coeffs[: harmonics + 1],
            "sin_n": sin_coeffs[: harmonics + 1],
        }
    )
    display(HTML(f"<h3>{title}</h3>"))
    display(df.round(4))


show_real_fourier_coefficients("Square wave coefficients", Sq(x))
show_real_fourier_coefficients("Triangle wave coefficients", Tr(x))

,n,cos_n,sin_n
0,0,-0.0002,0.0000
1,1,0.0003,0.9003
2,2,-0.0003,-0.0000
3,3,0.0003,0.3001
4,4,-0.0003,-0.0000
5,5,0.0003,0.1801
6,6,-0.0003,-0.0000
7,7,0.0003,0.1286
8,8,-0.0003,-0.0000
9,9,0.0003,0.1000


,n,cos_n,sin_n
0,0,-0.0,0.0000
1,1,-0.0,0.5732
2,2,-0.0,-0.0000
3,3,0.0,-0.0637
4,4,-0.0,-0.0000
5,5,-0.0,0.0229
6,6,-0.0,-0.0000
7,7,0.0,-0.0117
8,8,-0.0,-0.0000
9,9,-0.0,0.0071


## 8) Sounds

There are two useful sound workflows:

1. `play(...)` is the fastest way to audition a symbolic signal.
2. `plot.sound()` keeps audio attached to a plotted curve so the same sliders drive both the picture and the sound.

`play(...)` auto-normalizes. Figure-linked sound does **not**, so keep those expressions inside `[-1, 1]`.

In [29]:
display(play(0.8 * exp(-3 * x) * sin(2 * pi * 220 * x), (x, 0, 1.5)))
display(play(0.45 * sin(2 * pi * 220 * x) + 0.45 * sin(2 * pi * 224 * x), (x, 0, 1.5)))

The next figure links sound to a plot. Once `sound_generation_enabled(True)` is set, the legend shows a speaker button for each sound-capable plot. The same sliders now control both the waveform and the audio.

In [30]:
freq, mix = sp.symbols("freq mix")

fig_sound = Figure(x_range=(0, 0.02), y_range=(-1.05, 1.05), samples=2000)
fig_sound.show()

with fig_sound:
    set_title("Figure-linked sound: one plot, one speaker button")
    tone = plot(
        (0.65 - 0.35 * mix) * sin(2 * pi * freq * x) + 0.35 * mix * sin(4 * pi * freq * x),
        x,
        id="tone",
        label="tone",
        color="darkorange",
    )
    plot(
        0.65 * sin(2 * pi * freq * x),
        x,
        id="fundamental",
        label="fundamental",
        color="gray",
        opacity=0.25,
    )
    parameter(freq, value=220, min=110, max=880, step=10)
    parameter(mix, value=0.3, min=0.0, max=1.0, step=0.05)
    sound_generation_enabled(True)
    info(
        "<b>How to listen</b><br>The legend now has a speaker button for each sound-capable plot.<br>"
        "Click the button beside <code>tone</code>, or run <code>tone.sound(True)</code>.",
        id="sound-help",
    )

OneShotOutput()